# Criação da tabela Gold - ACURÁCIA

## Objetivos
- Verificação de acurácia de passes por time
- Verificação da acurácia de chutes ao gol por time
- Verificação da acurácia de passes

In [0]:
from pyspark.sql.functions import *

In [0]:
df_calculate_acuracy = (
    spark.read.table("api_football.silver.statistics")
    .select(
        "team_id",
        "team_name",
        "passes_accurate",
        "passes_total",
        "shots_on_goal",
        "shots_total",
        "shots_off_goal",
        "shots_blocked",
        "attacks",
    )
    .withColumn(
        "pass_accuracy_pct",round(col("passes_accurate") * 100 / col("passes_total"), 2),
    )
    .withColumn(
        "shots_accuracy_pct", round(col("shots_on_goal") * 100 / col("shots_total"), 2)
    )
    .withColumn(
        "shots_blocked_pct", round(col("shots_blocked") * 100 / col("shots_total"), 2)
    )
    .withColumn(
        "shots_off_pct", round(col("shots_off_goal") * 100 / col("shots_total"), 2)
    )
)
#display(df_calculate_acuracy)

In [0]:
df_team_acuracy = df_calculate_acuracy.groupBy("team_id", "team_name").agg(
    round(avg("shots_accuracy_pct"), 2).alias("shots_on_goal_pct"),
    round(avg("shots_blocked_pct"), 2).alias("shots_blocked_pct"),
    round(avg("shots_off_pct"), 2).alias("shots_off_goal_pct"),
    round(avg("pass_accuracy_pct"), 2).alias("pass_accuracy_pct")
).orderBy("team_name")

#isplay(df_team_acuracy)

In [0]:
df_team_acuracy.write.mode("overwrite").saveAsTable("api_football.gold.acuracy_statistics")

In [0]:
%sql
-- select
--   *
-- from
--   api_football.gold.acuracy_statistics